# Эмбеддинги описаний

Сначала выполнить `DatasetProcessing.ipynb`, чтобы получить `item_texts.parquet`.
Запускать из каталога `notebooks/`. В `embedding_jobs` указаны пары входного Parquet
и выходного pickle. Все варианты используют одну модель, токенизацию и masked mean
pooling; различаются только тексты. Маппинг сверяется с исходными описаниями.
Готовые выходные файлы не перезаписываются: для нового эксперимента задайте новое имя.


In [ ]:
import pickle

import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import LlamaModel, LlamaTokenizer


In [ ]:
original_texts_path = '../data/Beauty/item_texts.parquet'

# Выберите нужные пары вход/выход; готовые варианты повторно считать не нужно.
embedding_jobs = [
    ('../data/Beauty/enrichment/enriched_v2_cuda_20261104/summaries.parquet',
     '../data/Beauty/content_embeddings_enriched_v2.pkl'),
    # ('../data/Beauty/item_texts.parquet', '../data/Beauty/content_embeddings_original.pkl'),
    # ('../data/Beauty/enrichment/one_sentence_named_cuda_full_20261103_rerun1/summaries.parquet',
    #  '../data/Beauty/content_embeddings_enriched.pkl'),
]

model_name = 'huggyllama/llama-7b'
max_length = 1024
batch_size = 1


In [ ]:
from pathlib import Path

original = pd.read_parquet(original_texts_path).sort_values('item_id').reset_index(drop=True)
assert original['item_id'].tolist() == list(range(len(original)))
assert original['asin'].is_unique
assert embedding_jobs and len({output for _, output in embedding_jobs}) == len(embedding_jobs)
datasets = []
for input_path, output_path in embedding_jobs:
    assert not Path(output_path).exists(), f'Output already exists: {output_path}'
    frame = pd.read_parquet(input_path).sort_values('item_id').reset_index(drop=True)
    assert frame['item_id'].is_unique and frame['asin'].is_unique
    assert frame[['item_id', 'asin', 'text']].notna().all().all()
    assert frame['text'].map(lambda text: isinstance(text, str) and bool(text.strip())).all()
    assert original[['item_id', 'asin']].equals(frame[['item_id', 'asin']]), 'Different item mappings'
    datasets.append((frame, output_path))
    print(f'{input_path}: {len(frame)} items; mapping matches')


In [ ]:
tokenizer = LlamaTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

# Standard Transformers placement: adapts to available GPU/CPU memory, no fixed GPU IDs.
model = LlamaModel.from_pretrained(
    model_name, torch_dtype=torch.float32, device_map='auto', attn_implementation='eager',
).eval()
device = model.get_input_embeddings().weight.device


In [ ]:
class MyDataset:
    def __init__(self, data):
        self.items = list(data[['item_id', 'text']].itertuples(index=False, name=None))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item_id, text = self.items[idx]
        inputs = tokenizer(text, return_tensors='pt', max_length=max_length,
                           truncation=True, padding='max_length')
        return {
            'item_id': item_id,
            'input_ids': inputs['input_ids'][0],
            'attention_mask': inputs['attention_mask'][0],
        }


In [ ]:
for data, output_path in datasets:
    loader = DataLoader(MyDataset(data), batch_size=batch_size, shuffle=False, num_workers=0)
    result = {'item_id': [], 'embedding': []}

    for batch in tqdm(loader, desc=output_path):
        with torch.inference_mode():
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids=batch['input_ids'].to(device),
                            attention_mask=attention_mask, use_cache=False)
            embeddings = outputs.last_hidden_state
            mask = attention_mask.to(embeddings.device).unsqueeze(-1).to(embeddings.dtype)
            mean_embeddings = (embeddings * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            assert torch.isfinite(mean_embeddings).all()

        result['item_id'].extend(batch['item_id'].tolist())
        result['embedding'].extend(mean_embeddings.cpu().tolist())

    assert result['item_id'] == data['item_id'].tolist()
    with open(output_path, 'xb') as stream:
        pickle.dump(result, stream)
    print(f'Saved {len(result["item_id"])} embeddings to {output_path}')
